#Gold Layer in Databricks – Building Fact & Dimension Tables

##Load Silver Tables

In [0]:
customers_df  = spark.read.table('retial_project.silver.silver_customers')
orders_df = spark.read.table('retial_project.silver.silver_orders')

customers_df.display()
orders_df.display()

## Create Customer Dimension

In [0]:
dim_customer = customers_df.select("customer_id",
                    "customer_name",
                    "email",
                    "city",
                    "state",
                    "loyalty_points",
                    "status"                    
                    )
dim_customer.display()

In [0]:
dim_customer.write.format('delta').mode('overwrite').saveAsTable('retial_project.gold.gold_dim_customer')

## Create Fact Sales Table

In [0]:
fact_sales = orders_df.join(
    customers_df,
    orders_df.customer_id == customers_df.customer_id,
    "left"
).select(
    orders_df.order_id,
    orders_df.order_date,
    orders_df.customer_id,
    customers_df.customer_name,
    orders_df.product_id,
    orders_df.product_name,
    orders_df.category,
    orders_df.quantity,
    orders_df.unit_price,
    orders_df.discount,
    orders_df.total_amount,
    customers_df.city,
    customers_df.state,
    customers_df.gender,
    customers_df.loyalty_points
)

# fact_sales.display()


from pyspark.sql.functions import year , month , dayofmonth

order_enriched = fact_sales.\
    withColumn("Year",year("order_date")).\
    withColumn("Month",month("order_date")).\
    withColumn("Day",dayofmonth("order_date"))

order_enriched.display()


In [0]:

from pyspark.sql.functions import sum, countDistinct, avg, min,max

gold_df = order_enriched.groupBy(
    "product_id",
    "product_name",
    "customer_id",
    "customer_name",
    "category",
    "Year",
    "Month",
    "order_date",
    "gender",
    "city",
    "state",
    "loyalty_points"
).agg(
    sum("total_amount").alias("Total_revenue"),
    sum("quantity").alias("Total_quantity"),
    countDistinct("order_id").alias("Total_orders"),
    avg("total_amount").alias("Avg_revenue"),
    avg("quantity").alias("Avg_quantity"),
    avg("loyalty_points").alias("Avg_loyalty_points"),
    min("unit_price").alias("Min_price"),
    max("unit_price").alias("Max_price")
)
gold_df.display()

In [0]:

gold_df.write.format('delta').mode('overwrite').saveAsTable('retial_project.gold.gold_fact_Sales')